<a href="https://colab.research.google.com/github/angiecombs11-ops/PurdueHW/blob/main/7_03_Implementation_of_SGD_4_5_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
iris_data = load_iris()
X, y = iris_data.data, iris_data.target

In [3]:
encoder = OneHotEncoder(sparse_output = False)
y = encoder.fit_transform(y.reshape(-1, 1))

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [6]:
def softmax(z):
    if z.ndim == 1:
        z = z.reshape(1, -1)
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def compute_loss(X, y, weights, bias):
    predictions = softmax(np.dot(X, weights) + bias)
    loss = -np.mean(np.sum(y * np.log(predictions), axis=1))
    return loss

In [7]:
def sgd(X, y, weights, bias, learning_rate, epochs):
    for epoch in range(epochs):
        for i in range(X.shape[0]):
            # Compute the prediction
            z = np.dot(X[i], weights) + bias
            prediction = softmax(z).flatten()

            # Compute the error
            error = prediction - y[i]

            # Update the weights and bias
            weights -= learning_rate * np.outer(X[i], error)
            bias -= learning_rate * error

        # Optionally, print the loss for each epoch
        if epoch % 10 == 0:
            loss = compute_loss(X, y, weights, bias)
            print(f'Epoch {epoch}, Loss: {loss}')

    return weights, bias

In [10]:
# Define learning rate
learning_rate = 0.01
# Define number of epochs
epochs = 100

# Initialize weights and bias
weights = np.random.randn(X_train_sc.shape[1], y_train.shape[1])
bias = np.zeros(y_train.shape[1])

weights, bias = sgd(X_train_sc, y_train, weights, bias, learning_rate, epochs)

Epoch 0, Loss: 1.015570557154485
Epoch 10, Loss: 0.29001524279100316
Epoch 20, Loss: 0.22999560886896184
Epoch 30, Loss: 0.19544330965439052
Epoch 40, Loss: 0.17225912653370726
Epoch 50, Loss: 0.15557994625452884
Epoch 60, Loss: 0.14299930977119188
Epoch 70, Loss: 0.13316793009806843
Epoch 80, Loss: 0.12526888602083006
Epoch 90, Loss: 0.11877892914819048


In [11]:
# Evaluate the model
def predict(X, weights, bias):
    predictions = softmax(np.dot(X, weights) + bias)
    return np.argmax(predictions, axis=1)


# Make predictions
y_train_pred = predict(X_train_sc, weights, bias)
y_test_pred = predict(X_test_sc, weights, bias)

# Convert one-hot encoded targets back to labels for evaluation
y_train_true = np.argmax(y_train, axis=1)
y_test_true = np.argmax(y_test, axis=1)

# Compute accuracy
train_accuracy = accuracy_score(y_train_true, y_train_pred)
test_accuracy = accuracy_score(y_test_true, y_test_pred)

# Compute other evaluation metrics
train_conf_matrix = confusion_matrix(y_train_true, y_train_pred)
test_conf_matrix = confusion_matrix(y_test_true, y_test_pred)
train_class_report = classification_report(y_train_true, y_train_pred)
test_class_report = classification_report(y_test_true, y_test_pred)

# Print evaluation results
print(f'Training Accuracy: {train_accuracy}')
print(f'Testing Accuracy: {test_accuracy}')
print("Training Confusion Matrix:\n", train_conf_matrix)
print("Testing Confusion Matrix:\n", test_conf_matrix)
print("Training Classification Report:\n", train_class_report)
print("Testing Classification Report:\n", test_class_report)

Training Accuracy: 0.9666666666666667
Testing Accuracy: 1.0
Training Confusion Matrix:
 [[40  0  0]
 [ 0 38  3]
 [ 0  1 38]]
Testing Confusion Matrix:
 [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
Training Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        40
           1       0.97      0.93      0.95        41
           2       0.93      0.97      0.95        39

    accuracy                           0.97       120
   macro avg       0.97      0.97      0.97       120
weighted avg       0.97      0.97      0.97       120

Testing Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      